# Exploratory Data Analysis: Multilingual Toxic Comment Classification

This notebook conducts an in-depth exploratory data analysis (EDA) of toxic comment datasets, focusing on:
1. Class distribution & imbalance ratios
2. Text length and token distributions
3. Language distributions & zero-shot cross-lingual dynamics
4. Missing values, duplicates, and lexical patterns (obfuscation, punctuation, emojis)

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Add project root
sys.path.insert(0, str(Path.cwd().parent))
from preprocess import clean_text, prepare_dataframe
from config import DATASET_CONFIG

## 1. Load Dataset
We inspect either the full Jigsaw dataset if downloaded, or the synthetic verification dataset.

In [ ]:
dataset_path = DATASET_CONFIG.train_file if DATASET_CONFIG.train_file.exists() else DATASET_CONFIG.synthetic_file
print(f"Loading data from: {dataset_path}")
df_raw = pd.read_csv(dataset_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

## 2. Preprocess & Clean Text
We apply our non-destructive cleaning pipeline that preserves casing, punctuation, and emojis.

In [ ]:
df_clean = prepare_dataframe(
    df_raw,
    text_col=DATASET_CONFIG.text_col,
    label_col=DATASET_CONFIG.label_col,
    lang_col=DATASET_CONFIG.lang_col,
    drop_duplicates=True,
)
print(f"Cleaned dataset shape: {df_clean.shape}")
df_clean.head()

## 3. Class Imbalance Analysis
Toxicity datasets typically suffer from severe imbalance, where only 8%–15% of comments are toxic.

In [ ]:
class_counts = df_clean["label"].value_counts()
class_pcts = df_clean["label"].value_counts(normalize=True) * 100

print("Class Counts:")
for label, count in class_counts.items():
    name = "Toxic (1)" if label == 1 else "Non-toxic (0)"
    print(f"  {name}: {count} ({class_pcts[label]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#2ca02c", "#d62728"], ax=ax)
ax.set_xticklabels(["Non-toxic (0)", "Toxic (1)"])
ax.set_ylabel("Number of Comments")
ax.set_title("Class Distribution")
plt.tight_layout()
plt.show()

## 4. Sequence Length & Token Distribution
Understanding the distribution of comment lengths helps determine an optimal `max_length` truncation parameter for XLM-RoBERTa (e.g. 128 vs 256 tokens) to maximize GPU memory efficiency.

In [ ]:
df_clean["char_length"] = df_clean["clean_text"].str.len()
df_clean["approx_tokens"] = df_clean["clean_text"].str.split().str.len()

print("Character Length Summary:")
print(df_clean["char_length"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df_clean, x="approx_tokens", hue="label", kde=True, bins=30, ax=ax1, palette=["#2ca02c", "#d62728"])
ax1.set_title("Approximate Word Count Distribution")
ax1.set_xlabel("Number of Words")

sns.boxplot(data=df_clean, x="label", y="char_length", ax=ax2, palette=["#2ca02c", "#d62728"])
ax2.set_xticklabels(["Non-toxic (0)", "Toxic (1)"])
ax2.set_title("Character Length by Toxicity Label")
plt.tight_layout()
plt.show()

## 5. Language Breakdown
Inspect the languages present in the dataset.

In [ ]:
if "lang" in df_clean.columns:
    lang_counts = df_clean["lang"].value_counts()
    print("Language Breakdown:")
    print(lang_counts)
    
    plt.figure(figsize=(8, 4))
    sns.barplot(x=lang_counts.index, y=lang_counts.values, palette="viridis")
    plt.title("Comments per Language Code")
    plt.xlabel("Language Code")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()